# 00: Configuración del Proyecto

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la Universitat Jaume I**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |
| **Versión** | AU_UJI Dinámico (V2) |

---

> ℹ️ **Validación de datos profunda:** este notebook hace una verificación
> **rápida** de la existencia de los Excel originales (nivel N1).
> Para una validación **completa** (5 niveles: existencia, estructura de hojas,
> columnas obligatorias, tipos de datos y cruces de IDs), ejecutar el notebook
> `f0_validar_excel.ipynb` (Fase 0 — TAREA C).

---

## 🎯 ¿Qué hace este notebook?

Este notebook es el **punto de entrada** del proyecto. Se ejecuta **siempre primero**,
antes de cualquier fase. Hace 4 cosas:

1. **Configura** las rutas del proyecto (funciona en Local y en Google Colab)
2. **Verifica** que los archivos Excel originales están en su sitio (validación N1 simple)
3. **Crea** las carpetas de datos, HTML y notebooks si no existen
4. **Genera** el `index.html` principal del proyecto

## ⚠️ Requisitos

Solo necesita que existan los 2 archivos Excel originales en `data/00_raw/`:
- `datos_proyecto_sin_preinscrip.xlsx`
- `preinscripcion_si.xlsx`

## 📁 Estructura de carpetas de datos

| Carpeta | Fase | Contenido |
|---|---|---|
| `00_raw` | — | Excel originales (datos de la universidad) |
| `01_interim` | Fase 1 | Parquets individuales (1 por hoja del Excel) |
| `02_processed` | Fase 1 | df_alumno.parquet (todas las tablas unidas) |
| `03_features` | Fase 3 | dataset_final_tfm.parquet (dataset para ML) |
| `04_eda_final` | Fase 4 | Resultados de EDA final |
| `05_modelado` | Fase 5 | Modelos entrenados (.pkl) y métricas |
| `06_evaluacion` | Fase 6 | Resultados de interpretabilidad y fairness |
| `automl` | AutoML | Resultados de frameworks AutoML |

## 📁 Estructura de carpetas de notebooks

| Carpeta | Contenido |
|---|---|
| `fase0_configuracion` | Este notebook y orquestador maestro |
| `fase1_transformacion` | Limpieza y unificación de datos |
| `fase2_eda` | Análisis exploratorio de datos originales |
| `fase3_features` | Ingeniería de variables |
| `fase_automl` | Exploración de frameworks AutoML |
| `fase4_eda` | EDA final del dataset modelado |
| `fase5_modelado` | Modelado supervisado |
| `fase6_evaluacion` | Interpretabilidad, fairness y evaluación |
| `fase7_app` | Aplicación web (Streamlit, ver V2) |

## 🌐 Aplicación web

- **Versión 2 (esta):** https://tfm-abandono-dinamico.streamlit.app
- **Repositorio V2:** https://github.com/mortemj/AU_UJI_v2
- **Documentación V2:** https://mortemj.github.io/AU_UJI_v2/

## 📁 Salida

- `docs/html/index.html`


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
#
# Detecta entorno (Colab/Local), configura ROOT buscando 'src/',
# e importa TODAS las rutas de src.config — NUNCA hardcodeadas.
# ============================================================================

import sys
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# --- Detectar entorno ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto (rutas desde src.config, NUNCA hardcodeadas) ---
from src.config import (
    RUTA_RAW, RUTA_INTERIM, RUTA_PROCESSED, RUTA_FEATURES, RUTA_AUTOML,
    RUTA_HTML, RUTA_ASSETS, RUTA_NOTEBOOKS,
    EXCEL_PRINCIPAL, EXCEL_PREINSCRIPCION,
    DATASET_FINAL_PARQUET,
    AUTORA, EMAIL_UOC, EMAIL_UJI, GITHUB_REPO,
    COLORES, FASES_CONFIG,
    info_entorno
)

# --- Mostrar información ---
print('=' * 60)
print('CONFIGURACIÓN DEL PROYECTO')
print('=' * 60)
print(f'\nEntorno: {"Google Colab" if "google.colab" in sys.modules else "Local"}')
print(f'ROOT:    {ROOT}')
print(f'\nRutas de datos:')
print(f'  00_raw:        {RUTA_RAW}')
print(f'  01_interim:    {RUTA_INTERIM}')
print(f'  02_processed:  {RUTA_PROCESSED}')
print(f'  03_features:   {RUTA_FEATURES}')
print(f'  automl:        {RUTA_AUTOML}')

CONFIGURACIÓN DEL PROYECTO

Entorno: Local
ROOT:    c:\FF\AU_UJI_v2

Rutas de datos:
  00_raw:        c:\FF\AU_UJI_v2\data\00_raw
  01_interim:    c:\FF\AU_UJI_v2\data\01_interim
  02_processed:  c:\FF\AU_UJI_v2\data\02_processed
  03_features:   c:\FF\AU_UJI_v2\data\03_features
  automl:        c:\FF\AU_UJI_v2\data\automl


In [2]:
# ============================================================================
# CELDA 2: VERIFICAR ARCHIVOS ORIGINALES (OBLIGATORIOS)
# ============================================================================
#
# Los 2 archivos Excel son los datos originales de la universidad (UJI).
# Sin ellos no se puede ejecutar ninguna fase del proyecto.
# Las rutas vienen de EXCEL_PRINCIPAL y EXCEL_PREINSCRIPCION (src.config).
#
# NOTA: Para validación profunda (5 niveles: estructura, columnas, tipos,
# cruces de IDs), ejecutar `f0_validar_excel.ipynb` que usa el paquete
# `src.validacion`. Este notebook solo hace validación N1 (existencia).
# ============================================================================

print('=' * 60)
print('VERIFICACIÓN DE ARCHIVOS ORIGINALES')
print('=' * 60)

# Lista de archivos obligatorios (rutas de src.config)
ARCHIVOS_OBLIGATORIOS = [
    {
        'nombre': EXCEL_PRINCIPAL.name,
        'ruta': EXCEL_PRINCIPAL,
        'descripcion': 'Datos académicos principales (8 hojas)'
    },
    {
        'nombre': EXCEL_PREINSCRIPCION.name,
        'ruta': EXCEL_PREINSCRIPCION,
        'descripcion': 'Datos de preinscripción (solo alumnos matriculados)'
    },
]

# Verificar cada archivo
archivos_ok = True
for archivo in ARCHIVOS_OBLIGATORIOS:
    if archivo['ruta'].exists():
        tamano_mb = archivo['ruta'].stat().st_size / (1024 * 1024)
        print(f'\n  ✓ {archivo["nombre"]} ({tamano_mb:.1f} MB)')
        print(f'    {archivo["descripcion"]}')
    else:
        archivos_ok = False
        print(f'\n  ✗ ERROR: {archivo["nombre"]} NO ENCONTRADO')
        print(f'    Ruta esperada: {archivo["ruta"]}')
        print(f'    Este archivo es obligatorio para el proyecto.')

if archivos_ok:
    print(f'\n✓ Todos los archivos originales encontrados')
else:
    raise FileNotFoundError(
        f'Faltan archivos obligatorios en {RUTA_RAW}. '
        'Copia los archivos Excel originales a esa carpeta.'
    )

VERIFICACIÓN DE ARCHIVOS ORIGINALES

  ✓ datos_proyecto_sin_preinscrip.xlsx (27.8 MB)
    Datos académicos principales (8 hojas)

  ✓ preinscripcion_si.xlsx (23.4 MB)
    Datos de preinscripción (solo alumnos matriculados)

✓ Todos los archivos originales encontrados


In [3]:
# ============================================================================
# CELDA 3: CREAR CARPETAS
# ============================================================================
#
# Crea todas las carpetas del proyecto si no existen.
# Si una carpeta ya existe, no la toca (no borra contenido).
# ============================================================================

print('=' * 60)
print('CREACIÓN DE CARPETAS')
print('=' * 60)

# --- Carpetas de datos ---
carpetas_datos = [
    (RUTA_RAW,       'Datos originales (Excel)'),
    (RUTA_INTERIM,   'Parquets intermedios (Fase 1)'),
    (RUTA_PROCESSED, 'Dataset unificado (Fase 1)'),
    (RUTA_FEATURES,  'Features para ML (Fase 3)'),
    (RUTA_AUTOML,    'Dataset AutoML (Fase 3)'),
]

print('\nCarpetas de datos:')
for ruta, descripcion in carpetas_datos:
    ya_existia = ruta.exists()
    ruta.mkdir(parents=True, exist_ok=True)
    estado = 'ya existía' if ya_existia else 'CREADA'
    print(f'  ✓ {ruta.name:20s} ({estado}) — {descripcion}')

# --- Carpetas de HTML ---
carpetas_html = [
    RUTA_HTML,
    RUTA_HTML / 'fase1',
    RUTA_HTML / 'fase2',
    RUTA_HTML / 'fase3',
    RUTA_HTML / 'fase_automl',
    RUTA_HTML / 'fase4',
    RUTA_HTML / 'fase5',
    RUTA_HTML / 'fase6',
    RUTA_HTML / 'fase7',
]

print('\nCarpetas de HTML:')
for ruta in carpetas_html:
    ya_existia = ruta.exists()
    ruta.mkdir(parents=True, exist_ok=True)
    estado = 'ya existía' if ya_existia else 'CREADA'
    nombre = str(ruta.relative_to(RUTA_HTML)) if ruta != RUTA_HTML else 'html/'
    print(f'  ✓ {nombre:20s} ({estado})')

# --- Carpetas de notebooks ---
carpetas_nb = [
    'fase0_configuracion',
    'fase1_transformacion',
    'fase2_eda',
    'fase3_features',
    'fase_automl',
    'fase4_eda',
    'fase5_modelado',
    'fase6_evaluacion',
    'fase7_app',
]

print('\nCarpetas de notebooks:')
for nombre in carpetas_nb:
    ruta = RUTA_NOTEBOOKS / nombre
    ya_existia = ruta.exists()
    ruta.mkdir(parents=True, exist_ok=True)
    estado = 'ya existía' if ya_existia else 'CREADA'
    print(f'  ✓ {nombre:25s} ({estado})')

print(f'\n✓ Todas las carpetas verificadas')


CREACIÓN DE CARPETAS

Carpetas de datos:
  ✓ 00_raw               (ya existía) — Datos originales (Excel)
  ✓ 01_interim           (ya existía) — Parquets intermedios (Fase 1)
  ✓ 02_processed         (ya existía) — Dataset unificado (Fase 1)
  ✓ 03_features          (ya existía) — Features para ML (Fase 3)
  ✓ automl               (ya existía) — Dataset AutoML (Fase 3)

Carpetas de HTML:
  ✓ html/                (ya existía)
  ✓ fase1                (ya existía)
  ✓ fase2                (ya existía)
  ✓ fase3                (ya existía)
  ✓ fase_automl          (ya existía)
  ✓ fase4                (ya existía)
  ✓ fase5                (ya existía)
  ✓ fase6                (ya existía)
  ✓ fase7                (ya existía)

Carpetas de notebooks:
  ✓ fase0_configuracion       (ya existía)
  ✓ fase1_transformacion      (ya existía)
  ✓ fase2_eda                 (ya existía)
  ✓ fase3_features            (ya existía)
  ✓ fase_automl               (ya existía)
  ✓ fase4_eda              

In [4]:
# ============================================================================
# CELDA 4: GENERAR INDEX.HTML PRINCIPAL
# ============================================================================
#
# Genera la página principal del proyecto (docs/html/index.html)
# usando la función ruta_index = actualizar_index() de src/index_generator.py.
#
# La función lee dinámicamente:
#   - Excel originales (openpyxl, read_only) → tablas, registros raw
#   - df_alumno.parquet (si existe) → expedientes, período
#   - df_expediente_features.parquet (si existe) → variables, tasa abandono
#
# Los KPIs no disponibles aparecen translúcidos con "Pendiente".
#
# Esta misma función se puede llamar desde el último notebook de cada
# fase para mantener el index siempre actualizado:
#
#   from src.index_generator import actualizar_index
#   ruta_index = actualizar_index()
# ============================================================================

from src.index_generator import actualizar_index

ruta_index = actualizar_index()

🔄 Actualizando index.html...
  ✅ KPIs: Expedientes: 30.872 | Período: 2010-2020 | Variables: 24 | Abandono: 29,2% | AUC: 0,9564
  📊 Fases: Fase 5: ✅ | Fase 6: ✅ | Fase 7: ✅
  ✅ c:\FF\AU_UJI_v2\docs\html\index.html
  ✅ c:\FF\AU_UJI_v2\docs\html\fase7\fase7_index.html


In [5]:
# ============================================================================
# CELDA 5: RESUMEN
# ============================================================================

print('\n' + '=' * 60)
print('CONFIGURACIÓN COMPLETADA')
print('=' * 60)

# --- Estado de los datos ---
print('\nEstado de los datos:')
print(f'  ✓ Excel originales en {RUTA_RAW.name}/')

parquets_interim = list(RUTA_INTERIM.glob('*.parquet')) if RUTA_INTERIM.exists() else []
tiene_alumno   = DATASET_FINAL_PARQUET.exists()
tiene_features = (RUTA_FEATURES / 'df_expediente_features.parquet').exists()
tiene_automl   = (RUTA_AUTOML / 'df_exp_automl_target_final.parquet').exists()

print(f'  {"✓" if parquets_interim else "⚠"} Fase 1 interim: {len(parquets_interim)} parquets en {RUTA_INTERIM.name}/')
print(f'  {"✓" if tiene_alumno else "⚠"} Fase 1 final: df_alumno.parquet en {RUTA_PROCESSED.name}/')
print(f'  {"✓" if tiene_features else "⚠"} Fase 3: df_expediente_features.parquet en {RUTA_FEATURES.name}/')
print(f'  {"✓" if tiene_automl else "⚠"} AutoML: dataset en {RUTA_AUTOML.name}/')

print(f'\nArchivos generados:')
print(f'  ✓ {ruta_index}')

print(f'\n📌 Siguiente: Ejecutar los notebooks de la Fase 1')
print('=' * 60)


CONFIGURACIÓN COMPLETADA

Estado de los datos:
  ✓ Excel originales en 00_raw/
  ✓ Fase 1 interim: 9 parquets en 01_interim/
  ✓ Fase 1 final: df_alumno.parquet en 02_processed/
  ✓ Fase 3: df_expediente_features.parquet en 03_features/
  ⚠ AutoML: dataset en automl/

Archivos generados:
  ✓ c:\FF\AU_UJI_v2\docs\html\index.html

📌 Siguiente: Ejecutar los notebooks de la Fase 1


In [6]:
# ============================================================================
# CELDA 6: VERIFICACIÓN ANTI-HARDCODE
# ============================================================================
#
# Test de humo final: verifica que la cadena de imports y rutas del proyecto
# está bien conectada. Si algún check falla, hay un problema estructural
# que debe arreglarse antes de continuar con las fases.
# ============================================================================

print('=' * 60)
print('VERIFICACIÓN ANTI-HARDCODE')
print('=' * 60)

# 1. Verificar que src.config exporta todo lo necesario
from src.config import (
    RUTA_RAW, RUTA_INTERIM, RUTA_PROCESSED, RUTA_FEATURES, RUTA_AUTOML,
    RUTA_HTML, EXCEL_PRINCIPAL, EXCEL_PREINSCRIPCION, DATASET_FINAL_PARQUET,
    AUTORA, EMAIL_UOC, EMAIL_UJI, GITHUB_REPO, COLORES
)
print('  ✅ 1. Imports de src.config OK')

# 2. Verificar que las rutas son las nuevas (00_raw, 01_interim, 02_processed)
errores_rutas = []
for nombre, ruta in [('RAW', RUTA_RAW), ('INTERIM', RUTA_INTERIM), ('PROCESSED', RUTA_PROCESSED)]:
    s = str(ruta)
    if '01_raw' in s or '02_interim' in s or '03_processed' in s:
        errores_rutas.append(f'{nombre} apunta a ruta ANTIGUA: {s}')
if errores_rutas:
    print(f'  ❌ 2. Rutas antiguas detectadas: {errores_rutas}')
else:
    print('  ✅ 2. Rutas correctas (00_raw, 01_interim, 02_processed, …)')

# 3. Verificar que EXCEL_PRINCIPAL apunta al fichero esperado
if 'datos_proyecto' in str(EXCEL_PRINCIPAL).lower():
    print(f'  ✅ 3. EXCEL_PRINCIPAL = {EXCEL_PRINCIPAL.name}')
else:
    print(f'  ❌ 3. EXCEL_PRINCIPAL no apunta a datos_proyecto: {EXCEL_PRINCIPAL.name}')

# 4. Verificar que módulos auxiliares importan los mismos metadatos
try:
    from src.html.components import AUTORA as A_html
    from src.utils.files import AUTORA as A_files
    if A_html == A_files == AUTORA:
        print(f"  ✅ 4. components.py y files.py comparten AUTORA = '{AUTORA}'")
    else:
        print(f'  ⚠️  4. AUTORA inconsistente: config={AUTORA!r}, html={A_html!r}, files={A_files!r}')
except ImportError as e:
    print(f'  ⚠️  4. No se pudo verificar AUTORA en módulos auxiliares: {e}')

# 5. Verificar que el generador de index.html es importable y callable
try:
    from src.index_generator import actualizar_index
    if callable(actualizar_index):
        print('  ✅ 5. src.index_generator.actualizar_index es importable y callable')
    else:
        print('  ❌ 5. actualizar_index no es callable')
except ImportError as e:
    print(f'  ❌ 5. No se pudo importar src.index_generator: {e}')

print('\n' + '=' * 60)
print('Verificación finalizada')
print('=' * 60)


VERIFICACIÓN ANTI-HARDCODE
  ✅ 1. Imports de src.config OK
  ✅ 2. Rutas correctas (00_raw, 01_interim, 02_processed, …)
  ✅ 3. EXCEL_PRINCIPAL = datos_proyecto_sin_preinscrip.xlsx
  ✅ 4. components.py y files.py comparten AUTORA = 'María José Morte'
  ✅ 5. src.index_generator.actualizar_index es importable y callable

Verificación finalizada
